run_all.py -- run the whole study:  python run_all.py --seeds 10

Results (CSV + PNG) are written to ./results.  If the UCI files are not found in
./data and cannot be downloaded, the synthetic fallback is used (and announced).

In [11]:
import sys
import os

# Ensure the current directory is in Python's path to find local modules like 'experiments'
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

import pandas as pd
import experiments as ex
import plot # Fixed from 'import plot' to 'import plots'

pd.set_option("display.width", 200, "display.max_columns", 30)

def main(
    data_dir="data",
    out="results",
    seeds=10,
    rounds=40,
    synthetic=False,
    skip_he=False,
    only=None
):
    if only is None:
        only = ["algorithms", "privacy", "security", "communication", "framework"]
        
    os.makedirs(out, exist_ok=True)
    seed_list = list(range(seeds))
    
    # Load real data (synthetic=False ensures real UCI dataset files are loaded)[cite: 1]
    raw, desc = ex.load_raw(data_dir, synthetic)
    print(f"\n=== DATA: {desc} ===")

    def save(name, df, hist=None, conv_names=None):
        df.to_csv(f"{out}/{name}.csv", index=False)
        print(f"\n--- {name} ---")
        print(ex.summarise(df))
        if hist and conv_names:
            plots.plot_convergence(hist, conv_names, "auc", f"{out}/{name}_auc.png",
                                   "global test AUC-ROC")

    if "algorithms" in only:
        df, h = ex.exp_algorithms(raw, seed_list, rounds)
        save("exp1_algorithms", df, h, list(df.name.unique()))
        print(ex.significance(df, "FedAvg"))
    if "privacy" in only:
        df, h = ex.exp_privacy(raw, seed_list, rounds, with_he=not skip_he)
        save("exp2_privacy", df)
        plots.plot_privacy_utility(df, f"{out}/exp2_privacy_utility.png")
    if "security" in only:
        df, h = ex.exp_security(raw, seed_list, rounds)
        df.to_csv(f"{out}/exp3_security.csv", index=False)
        print("\n--- exp3_security (mean F1) ---")
        print(df.pivot_table(index="aggregator", columns="attack", values="f1").round(3))
        print("\n--- adversarial accuracy under FGSM ---")
        print(df[df.attack == "none"].groupby("aggregator")["adv_accuracy"].mean().round(3))
    if "communication" in only:
        df, h = ex.exp_communication(raw, seed_list, rounds)
        save("exp4_communication", df, h, list(df.name.unique()))
    if "framework" in only:
        df, h = ex.exp_framework(raw, seed_list, rounds)
        cols = ex.METRICS + ["conv_rounds", "comm_kb", "time_s", "epsilon",
                             "utility_loss_%", "poison_retention_%"]
        save("exp5_framework", df, h, list(df.name.unique()))
        print(df.groupby("name", sort=False)[cols].mean().round(3))
        print("\nSignificance vs FedAvg (F1):")
        print(ex.significance(df, "FedAvg", "f1"))
        print("\nSignificance vs FedProx (AUC):")
        print(ex.significance(df, "FedProx", "auc"))
        plots.plot_bars(df, "f1", f"{out}/exp5_f1.png", "F1 by configuration")

if __name__ == "__main__":
    main(data_dir="data", out="results", synthetic=False)

AttributeError: module 'experiments' has no attribute 'load_raw'

In [5]:
!pip install experiments

  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached python_dotenv-1.2.4-py3-none-any.whl.metadata (29 kB)
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   ------------------- -------------------- 1.0/2.2 MB 1.2 MB/s eta 0:00:01
   ---------------------------- ----------- 1.6/2.2 MB 1.6 MB/s eta 0:00:01
   -------------------------------------- - 2.1/2.2 MB 1.8 MB/s eta 0:00:01
   -------------------------------------- - 2.1/2.2 MB 1.8 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 1.3 MB/s eta 0:00:00
Using cached python_dotenv-1.2.4-py3-none-any.whl (23 kB)
Using cached typing_extensions-4.16.0-py3-none-any.whl (45 kB)
Using

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires protobuf<7,>=3.20, but you have protobuf 7.36.2 which is incompatible.


In [8]:
!pip install plot

  DEPRECATION: Building 'typing' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'typing'. Discussion can be found at https://github.com/pypa/pip/issues/6334



  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26420 sha256=783dd3b0f234d88ce055d21a4f02a30be0e4ad04294db4f1ddcc5411f1fe5e32
  Stored in directory: c:\users\snigd\appdata\local\pip\cache\wheels\5d\1b\f4\fbdc2bc618f0ba7aec7295d67f62d40e6027a6ce14df9ec3c8
Successfully built typing

   -------------------- ------------------- 1/2 [plot]
   -------------------- ------------------- 1/2 [plot]
   -------------------- ------------------- 1/2 [plot]
   -------------------- ------------------- 1/2 [plot]
   -------------------- ------------------- 1/2 [plot]
   ---------------------------------------- 2/2 [plot]

